In [1]:
import tensorflow as tf
import tensorflow_datasets as tfds
import datetime
import matplotlib.pyplot as plt
import numpy as np
import itertools
import io
import sklearn.metrics

#### Downloading and Preprocessing the data

In [2]:
BUFFER_SIZE = 70_000
BATCH_SIZE = 128
NUM_EPOCHS = 20

In [3]:
mnist_dataset, mnist_info = tfds.load(name = 'mnist', with_info = True, as_supervised=True)

In [4]:
mnist_train, mnist_test = mnist_dataset['train'], mnist_dataset['test']

In [5]:
def scale(image,label):
    image = tf.cast(image,tf.float32)
    image /= 255.
    return image,label


In [6]:
train_and_validation_data = mnist_train.map(scale)
test_data = mnist_test.map(scale)

In [7]:
num_validation_samples = 0.1 * mnist_info.splits['train'].num_examples
num_validation_samples = tf.cast(num_validation_samples,tf.int64)
num_test_samples = mnist_info.splits['test'].num_examples
num_test_samples = tf.cast(num_test_samples,tf.int64)

In [8]:
train_and_validation_data = train_and_validation_data.shuffle(BUFFER_SIZE)
train_data = train_and_validation_data.skip(num_validation_samples)
validation_data = train_and_validation_data.take(num_validation_samples)

In [9]:
train_data = train_data.batch(BATCH_SIZE)

In [10]:
validation_data = validation_data.batch(num_validation_samples)
test_data = test_data.batch(num_test_samples)

In [11]:
for images,labels in validation_data:
    images_val = images.numpy()
    labels_val = labels.numpy()

I0000 00:00:1787163889.810184 1239137 tf_record_dataset_op.cc:396] The default buffer size is 262144, which is overridden by the user specified `buffer_size` of 8388608


#### Create the model and train it

In [12]:
model = tf.keras.models.Sequential([
    tf.keras.Input(shape=(28,28,1)),
    tf.keras.layers.Conv2D(50,5,activation='relu'),
    tf.keras.layers.MaxPooling2D(pool_size=(2,2)),
    tf.keras.layers.Conv2D(50,3,activation='relu'),
    tf.keras.layers.MaxPooling2D(pool_size=(2,2)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(10)
])

In [13]:
model.summary(line_length=75)

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┓
┃ Layer (type)                   ┃ Output Shape            ┃      Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                │ (None, 24, 24, 50)      │        1,300 │
├────────────────────────────────┼─────────────────────────┼──────────────┤
│ max_pooling2d (MaxPooling2D)   │ (None, 12, 12, 50)      │            0 │
├────────────────────────────────┼─────────────────────────┼──────────────┤
│ conv2d_1 (Conv2D)              │ (None, 10, 10, 50)      │       22,550 │
├────────────────────────────────┼─────────────────────────┼──────────────┤
│ max_pooling2d_1 (MaxPooling2D) │ (None, 5, 5, 50)        │            0 │
├────────────────────────────────┼─────────────────────────┼──────────────┤
│ flatten (Flatten)              │ (None, 1250)            │            0 │
├────────────────────────────────┼─────────────────────────┼──────────────┤
│ dense (Dense)                  │ (None, 10)              │       12,510 │
└────────────────────────────────┴─────────────────────────┴──────────────┘

 Total params: 36,360 (142.03 KB)

 Trainable params: 36,360 (142.03 KB)

 Non-trainable params: 0 (0.00 B)

First dimension in all the layers is None because all data is batched. Thus we don't pass a single image of size 28*28*1 but actually passes a batch of thousand images all at once. That's supposed to be the first dimension, however model does not know our batch size just yet as we have not trained it. So, it leaves this dimension as None.    
      
Trainable parameters are just weights of our network. The parameters that the model is trying to learn. Sometimes, however, you may need to multiply your input by a constant number, say 2.6 and you would want that to happen everytime. In other words, this parameter should not be changed during the learning process. These kinds of parameters are non-trainable.   

In [14]:
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
model.compile(optimizer='adam',loss=loss_fn,metrics=['accuracy'])

In [15]:
log_dir = "logs/fit/"+"run-1"

In [16]:
def plot_confusion_matrix(cm, class_names):

    figure = plt.figure(figsize=(12, 12))
    plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    plt.title("Confusion matrix")
    plt.colorbar()
    tick_marks = np.arange(len(class_names))
    plt.xticks(tick_marks, class_names, rotation=45)
    plt.yticks(tick_marks, class_names)

    # Normalize the confusion matrix.
    cm = np.around(cm.astype('float') / cm.sum(axis=1)[:, np.newaxis], decimals=2)

    # Use white text if squares are dark; otherwise black.
    threshold = cm.max() / 2.
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        color = "white" if cm[i, j] > threshold else "black"
        plt.text(j, i, cm[i, j], horizontalalignment="center", color=color)

    plt.tight_layout()
    plt.ylabel('True label')
    plt.xlabel('Predicted label')

    return figure

In [17]:
def plot_to_image(figure):
    """Converts the matplotlib plot specified by 'figure' to a PNG image and
    returns it. The supplied figure is closed and inaccessible after this call."""

    # Save the plot to a PNG in memory.
    buf = io.BytesIO()
    plt.savefig(buf, format='png')

    # Closing the figure prevents it from being displayed directly inside the notebook.
    plt.close(figure)

    buf.seek(0)

    # Convert PNG buffer to TF image
    image = tf.image.decode_png(buf.getvalue(), channels=4)

    # Add the batch dimension
    image = tf.expand_dims(image, 0)

    return image

In [18]:
# Define a file writer variable for logging purposes
file_writer_cm = tf.summary.create_file_writer(log_dir + '/cm')

def log_confusion_matrix(epoch, logs):
    # Use the model to predict the values from the validation dataset.
    test_pred_raw = model.predict(images_val)
    test_pred = np.argmax(test_pred_raw, axis=1)

    # Calculate the confusion matrix.
    cm = sklearn.metrics.confusion_matrix(labels_val, test_pred)

    # Log the confusion matrix as an image summary.
    figure = plot_confusion_matrix(cm, class_names=['0','1','2','3','4','5','6','7','8','9'])
    cm_image = plot_to_image(figure)

    # Log the confusion matrix as an image summary.
    with file_writer_cm.as_default():
        tf.summary.image("Confusion Matrix", cm_image, step=epoch)

In [19]:
cm_callback = tf.keras.callbacks.LambdaCallback(on_epoch_end=log_confusion_matrix)
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir,histogram_freq=1,profile_batch=0)

In [20]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    mode='auto',
    min_delta=0,
    patience=2,
    verbose=0,
    restore_best_weights=True
)
# Here these parameters define that program should stop the training process when the validation loss starts to increase for two subsequent 
# epochs because patience=2.

In [21]:
model.fit(
    train_data,
    epochs = NUM_EPOCHS,
    callbacks = [tensorboard_callback,cm_callback,early_stopping],
    validation_data=validation_data,
    verbose =2 #this specifies what to print out during the training process. verbose = 2 means it will print info only at the end 
    # of each epoch while verbose = 1 displays progress bar for every batch
)

Epoch 1/20


/Users/ankit/Workspace/Projects/ankit-github/AI-ML-DS/53.Convolutional-Neural-Networks-with-TensorFlow-in-Python/.venv/lib/python3.13/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


188/188 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
422/422 - 9s - 21ms/step - accuracy: 0.9235 - loss: 0.2615 - val_accuracy: 0.9677 - val_loss: 0.0971
Epoch 2/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
422/422 - 8s - 19ms/step - accuracy: 0.9787 - loss: 0.0716 - val_accuracy: 0.9785 - val_loss: 0.0672
Epoch 3/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
422/422 - 9s - 20ms/step - accuracy: 0.9845 - loss: 0.0521 - val_accuracy: 0.9887 - val_loss: 0.0422
Epoch 4/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
422/422 - 8s - 20ms/step - accuracy: 0.9868 - loss: 0.0426 - val_accuracy: 0.9880 - val_loss: 0.0443
Epoch 5/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
422/422 - 9s - 20ms/step - accuracy: 0.9888 - loss: 0.0354 - val_accuracy: 0.9932 - val_loss: 0.0236
Epoch 6/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
422/422 - 8s - 20ms/step - accuracy: 0.9905 - loss: 0.0305 - val_accuracy: 0.9893 - val_loss: 0.0290
Epoch 7/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
422/422 - 9s - 21ms/step - accuracy: 0.99

#### Visualizing in Tensorboard

In [24]:
%reload_ext tensorboard 
%tensorboard --logdir "logs/fit"

Reusing TensorBoard on port 6006 (pid 14323), started 0:00:12 ago. (Use '!kill 14323' to kill it.)